# Yemen MarketShield AI — التحليل الأولي

**مشروع بحثي:** دراسة التنبؤ بصدمات أسعار الغذاء في اليمن باستخدام أسعار الغذاء والصرف والطاقة.

هذا الدفتر يقرأ ملفات البنك الدولي الثلاثة، ويفحص مفاتيحها وجودتها، ثم يجري تجربتين أوليتين: (1) تنبؤ بسعر القمح المقدّر للشهر، و(2) تصنيف ما إذا كان سعر القمح المقدّر سيرتفع بأكثر من 5% في الشهر التالي.

> **تنبيه علمي مهم:** ملفات Real-Time Prices التابعة للبنك الدولي تجمع بين قياسات مباشرة وتقديرات تعلم آلة للقيم المفقودة، كما أن أعمدة `o_`, `h_`, `l_`, `c_` هي تقديرات شهرية Open/High/Low/Close. لذلك فإن `c_wheat` هدفه في التجربة هو *تقدير الإغلاق الشهري في قاعدة البيانات*، وليس سعرًا ميدانيًا مستقلًا مؤكدًا. سنستخدم النتائج لفحص المنهجية، ولا نزعم أن النموذج يتنبأ بالسعر الحقيقي في كل سوق دون تقييم مستقل. المصدر الرسمي يشرح ذلك هنا: [وصف بيانات الغذاء](https://microdata.worldbank.org/catalog/4508) و[وصف بيانات الطاقة](https://microdata.worldbank.org/catalog/6133/study-description).

إصدار الملفات المستخدم في هذه الدراسة هو `2026-03-24`، والفترة حتى مارس 2026. احتفظ بالإصدار نفسه عند إعادة التجارب لتظل النتائج قابلة للتكرار.

## 1. تحميل ملفات ZIP الثلاثة

In [ ]:
# In Google Colab, upload the three official ZIP archives when prompted.
# In local Jupyter, place those same ZIP files beside this notebook before running it.
from pathlib import Path
import io, zipfile, os
import pandas as pd
import numpy as np

EXPECTED = {
    "food": "YEM_RTFP_mkt_2009_2026-03-24.csv",
    "fx": "YEM_RTFX_mkt_2009_2026-03-24.csv",
    "energy": "YEM_RTEP_mkt_2009_2026-03-24.csv",
}

try:
    from google.colab import files
    print("Upload the three World Bank ZIP archives listed in the project README.")
    uploaded = files.upload()
except ImportError:
    expected_zips = [
        Path("YEM_RTFP_mkt_2009_2026-03-24.zip"),
        Path("YEM_RTFX_mkt_2009_2026-03-24.zip"),
        Path("YEM_RTEP_mkt_2009_2026-03-24.zip"),
    ]
    missing = [str(p) for p in expected_zips if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            "Missing source ZIP files: " + ", ".join(missing) +
            ". Download them from the World Bank catalog links in README.md."
        )
    uploaded = {p.name: p.read_bytes() for p in expected_zips}

def read_csv_from_uploaded_zip(expected_csv):
    matches = []
    for filename, contents in uploaded.items():
        if filename.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(contents)) as zf:
                csv_names = [n for n in zf.namelist() if n.lower().endswith('.csv')]
                for name in csv_names:
                    if os.path.basename(name) == expected_csv:
                        with zf.open(name) as f:
                            matches.append(pd.read_csv(f, low_memory=False))
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one ZIP containing {expected_csv}; found {len(matches)}")
    return matches[0]

food = read_csv_from_uploaded_zip(EXPECTED['food'])
fx = read_csv_from_uploaded_zip(EXPECTED['fx'])
energy = read_csv_from_uploaded_zip(EXPECTED['energy'])
for df in (food, fx, energy):
    df['price_date'] = pd.to_datetime(df['price_date'])
print('Food:', food.shape, '| FX:', fx.shape, '| Energy:', energy.shape)


## 2. فحص بنية البيانات والتغطية

In [ ]:
def quick_summary(name, df):
    print(f"\n{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print('Period:', df['price_date'].min().date(), 'to', df['price_date'].max().date())
    print('Unique geo_id:', df['geo_id'].nunique())
    print('Unique market names:', df['mkt_name'].nunique())
    print('Duplicate (geo_id, price_date) keys:', df.duplicated(['geo_id','price_date']).sum())
    print('Markets:', sorted(df['mkt_name'].dropna().unique()))

quick_summary('Food', food)
quick_summary('Exchange rate', fx)
quick_summary('Energy', energy)

**ملاحظة عن عدد الأسواق:** تحتوي البيانات على 25 قيمة `geo_id`، وتشمل 24 سوقًا محليًا بالإضافة إلى سجل `Market Average`، ولذلك لا نعامل المتوسط الوطني بوصفه سوقًا محليًا عاديًا عند تحليل الأسواق الفردية.

## 3. التأكد من إمكانية الدمج

In [ ]:
KEY = ['geo_id', 'price_date']
key_sets = {
    'food': set(map(tuple, food[KEY].astype(str).to_numpy())),
    'fx': set(map(tuple, fx[KEY].astype(str).to_numpy())),
    'energy': set(map(tuple, energy[KEY].astype(str).to_numpy())),
}
print('Food = FX keys:', key_sets['food'] == key_sets['fx'])
print('Food = Energy keys:', key_sets['food'] == key_sets['energy'])
print('Shared key count:', len(key_sets['food'] & key_sets['fx'] & key_sets['energy']))

# نحتفظ فقط بالأعمدة اللازمة للتجارب الأولى لتجنب تكرار أعمدة التعريف.
food_keep = ['geo_id','price_date','mkt_name','adm1_name','lat','lon','c_wheat','wheat','c_wheat_flour','c_rice','c_sugar','c_food_price_index','trust_wheat','data_coverage_recent','index_confidence_score','spatially_interpolated']
fx_keep = ['geo_id','price_date','c_exchange_rate_unofficial','exchange_rate_unofficial','trust_exchange_rate_unofficial','c_milling_cost_wheat']
energy_keep = ['geo_id','price_date','c_fuel_diesel','fuel_diesel','c_fuel_gas','fuel_gas','c_fuel_petrol_gasoline','fuel_petrol_gasoline','trust_fuel_diesel','spatially_interpolated']
merged = food[[c for c in food_keep if c in food.columns]].merge(
    fx[[c for c in fx_keep if c in fx.columns]], on=KEY, how='left', validate='one_to_one', suffixes=('','_fx')
).merge(
    energy[[c for c in energy_keep if c in energy.columns]], on=KEY, how='left', validate='one_to_one', suffixes=('','_energy')
)
print('Merged shape:', merged.shape)
print('Missing values in key estimates:')
print(merged[['c_wheat','c_exchange_rate_unofficial','c_fuel_diesel']].isna().sum())
merged.head()

## 4. رسم أولي لأسعار القمح

In [ ]:
import matplotlib.pyplot as plt
market = 'Market Average'
plot_df = merged.loc[merged['mkt_name'].eq(market)].sort_values('price_date')
if plot_df.empty:
    # إن لم يكن متوسط السوق موجودًا نختار السوق الأكثر اكتمالًا
    market = merged['mkt_name'].value_counts().index[0]
    plot_df = merged.loc[merged['mkt_name'].eq(market)].sort_values('price_date')
plt.figure(figsize=(12,4))
plt.plot(plot_df['price_date'], plot_df['c_wheat'], label='c_wheat (World Bank close estimate)')
plt.title(f'Estimated wheat price over time — {market}')
plt.xlabel('Date'); plt.ylabel('Price in recorded currency (YER)')
plt.grid(True, alpha=0.25); plt.legend(); plt.tight_layout(); plt.show()
print('Market used:', market)

## 5. التجربة الأولى: توقع مستوى سعر القمح للشهر

نستخدم تقسيمًا زمنيًا: التدريب قبل يناير 2022 والاختبار من يناير 2022 حتى آخر تاريخ متاح. النموذج لا يرى ميزات من الشهر المستهدف إلا السعر السابق والخصائص المحسوبة من الماضي. خط الأساس `lag-1` يفترض أن سعر الشهر القادم يساوي آخر تقدير إغلاق متاح؛ وهذا خط أساس بسيط لكنه قوي في كثير من السلاسل السعرية.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model_data = merged.sort_values(['geo_id','price_date']).copy()
lag_series = ['c_wheat','c_wheat_flour','c_rice','c_sugar','c_exchange_rate_unofficial','c_fuel_diesel','c_fuel_gas','c_fuel_petrol_gasoline','c_milling_cost_wheat']
for col in lag_series:
    if col in model_data.columns:
        for lag in (1,2,3,12):
            model_data[f'{col}_lag{lag}'] = model_data.groupby('geo_id')[col].shift(lag)
model_data['month_num'] = model_data['price_date'].dt.month
model_data = model_data.dropna(subset=['c_wheat','c_wheat_lag1']).copy()
train = model_data[model_data['price_date'] < '2022-01-01']
test = model_data[model_data['price_date'] >= '2022-01-01']
features = [f'{c}_lag{lag}' for c in lag_series for lag in (1,2,3,12) if f'{c}_lag{lag}' in model_data.columns] + ['month_num']

def regression_metrics(y_true, y_pred):
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
            'WAPE': np.sum(np.abs(y_true-y_pred)) / np.sum(np.abs(y_true))}

results = []
results.append({'Model':'Naive: previous month', **regression_metrics(test['c_wheat'].to_numpy(), test['c_wheat_lag1'].to_numpy())})
# تعلّم التغير من الشهر السابق ثم نضيفه إلى سعر الشهر السابق
for name, estimator in [
    ('Random Forest on price level', RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.8, random_state=42, n_jobs=-1)),
    ('Gradient Boosting on price change', GradientBoostingRegressor(n_estimators=250, learning_rate=0.03, max_depth=2, loss='huber', random_state=42)),
]:
    target_col = 'c_wheat' if name.startswith('Random') else 'delta_target'
    if target_col == 'delta_target':
        model_data['delta_target'] = model_data['c_wheat'] - model_data['c_wheat_lag1']
        train = model_data[(model_data['price_date'] < '2022-01-01') & model_data['delta_target'].notna()]
        test = model_data[(model_data['price_date'] >= '2022-01-01') & model_data['delta_target'].notna()]
    else:
        train = model_data[model_data['price_date'] < '2022-01-01']
        test = model_data[model_data['price_date'] >= '2022-01-01']
    pipe = make_pipeline(SimpleImputer(strategy='median'), estimator)
    pipe.fit(train[features], train[target_col])
    raw_pred = pipe.predict(test[features])
    pred = test['c_wheat_lag1'].to_numpy() + raw_pred if target_col == 'delta_target' else raw_pred
    results.append({'Model':name, **regression_metrics(test['c_wheat'].to_numpy(), pred)})

pd.DataFrame(results).round(3)

### تفسير النتائج

لا نختار النموذج الأكثر تعقيدًا تلقائيًا. إذا كان خط الأساس (السعر السابق) أفضل، فهذا يعني أن نماذجنا الأولية لم تثبت بعد قيمة إضافية للتنبؤ بمستوى السعر. عندئذٍ ندرس التنبؤ بالتغيرات أو صدمات الأسعار، ونعيد التجربة بطريقة أكثر ملاءمة.

## 6. التجربة الثانية: إنذار مبكر بارتفاع يزيد على 5% في الشهر التالي

تعريف تجريبي فقط: نسمّي الحالة `shock_gt5` إذا زاد `c_wheat` في الشهر التالي بأكثر من 5% مقارنة بالشهر الحالي. هذا الحد اختيار أولي للاختبار، وليس تعريفًا اقتصاديًا رسميًا للأزمة. سنعرض كذلك معدل الحالات الإيجابية، حتى لا نخفي اختلال توازن الفئات.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

cls = merged.sort_values(['geo_id','price_date']).copy()
feature_series = ['c_wheat','c_wheat_flour','c_rice','c_sugar','c_food_price_index','c_exchange_rate_unofficial','c_fuel_diesel','c_fuel_gas','c_fuel_petrol_gasoline']
for col in feature_series:
    if col in cls.columns:
        for lag in (1,2,3,12):
            cls[f'{col}_lag{lag}'] = cls.groupby('geo_id')[col].shift(lag)
cls['next_c_wheat'] = cls.groupby('geo_id')['c_wheat'].shift(-1)
cls['next_date'] = cls.groupby('geo_id')['price_date'].shift(-1)
month_gap = (cls['next_date'].dt.year-cls['price_date'].dt.year)*12 + (cls['next_date'].dt.month-cls['price_date'].dt.month)
cls = cls[month_gap.eq(1)].copy() # لا نقارن إلا مع الشهر التالي مباشرة
cls['next_pct_change'] = cls['next_c_wheat']/cls['c_wheat'] - 1
cls['shock_gt5'] = (cls['next_pct_change'] > 0.05).astype(int)
cls['month_num'] = cls['price_date'].dt.month
numeric_features = [c for c in feature_series if c in cls.columns] + [f'{c}_lag{lag}' for c in ['c_wheat','c_wheat_flour','c_rice','c_exchange_rate_unofficial','c_fuel_diesel','c_fuel_gas','c_fuel_petrol_gasoline'] for lag in (1,2,3,12) if f'{c}_lag{lag}' in cls.columns] + ['month_num']
categorical_features = ['geo_id']
train = cls[cls['price_date'] < '2022-01-01']
test = cls[cls['price_date'] >= '2022-01-01']
X_train = train[numeric_features + categorical_features]
y_train = train['shock_gt5']
X_test = test[numeric_features + categorical_features]
y_test = test['shock_gt5']
preprocess = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric_features),
    ('market', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])
classifier = make_pipeline(preprocess, LogisticRegression(max_iter=2000, class_weight='balanced'))
classifier.fit(X_train, y_train)
prob = classifier.predict_proba(X_test)[:,1]
pred = (prob >= 0.5).astype(int)
print('Train rows:', len(train), '| Test rows:', len(test))
print('Positive rate — train:', round(y_train.mean(),3), '| test:', round(y_test.mean(),3))
print('Balanced accuracy:', round(balanced_accuracy_score(y_test,pred),3))
print('Precision:', round(precision_score(y_test,pred,zero_division=0),3))
print('Recall:', round(recall_score(y_test,pred,zero_division=0),3))
print('F1:', round(f1_score(y_test,pred,zero_division=0),3))
print('ROC-AUC:', round(roc_auc_score(y_test,prob),3))
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_test,pred))

### تفسير النتائج وحدود الاستخدام

الإنذار التجريبي قد يطلق إنذارات كاذبة؛ لذلك لا يجوز عرضه للمستخدمين بوصفه إنذارًا تشغيليًا قبل اختبار عتبات متعددة، والتحقق من البيانات المرصودة، ومعايرة الاحتمالات، وإجراء تقييم زمني ومكاني إضافي. وكل ما اختبرناه هنا هو ارتفاع `c_wheat` المقدّر، لا وقوع أزمة غذائية لدى السكان.

## 7. الخطوات التالية للمشروع

1. قراءة ملفات `ticker information` و`information on model` لفهم رموز المتغيرات ومؤشرات الثقة.
2. فصل الأسعار الخام المتاحة (مثل `wheat`) عن تقديرات `c_wheat`، وتصميم تحقق مستقل على السجلات المرصودة التي تتوافر لها مقارنة عادلة.
3. تقييم عدة عتبات للصدمة (+3%، +5%، +10%) وتحديدها مسبقًا في خطة البحث.
4. اختبار التعميم على أسواق لم تدخل التدريب، وليس فقط أشهر لاحقة في الأسواق نفسها.
5. كتابة تقرير يذكر أن البنك الدولي يدمج قياسات مباشرة مع تقديرات تعلم آلة للأسعار المفقودة، وأن تقديرات RTP قد تُراجع وتُحدّث. [المصدر الرسمي](https://microdata.worldbank.org/catalog/6133/study-description).

**مصادر البيانات:**
- [أسعار الغذاء — World Bank Microdata Library](https://microdata.worldbank.org/catalog/4508)
- [أسعار الصرف — World Bank Microdata Library](https://microdata.worldbank.org/catalog/6159)
- [أسعار الطاقة — World Bank Microdata Library](https://microdata.worldbank.org/catalog/6133)